In [1]:
# ==========================================================
# ACTIVIDAD 16. Preprocesamiento: recorte de extremos y escalado
# Los límites (percentiles 1 y 99) y el escalador se AJUSTAN solo
# con las empresas de entrenamiento en el bloque de entrenamiento,
# y se APLICAN sin cambios a todas las filas (sin fuga de información).
# ==========================================================
from pathlib import Path

import joblib
import pandas as pd
from google.colab import drive
from sklearn.preprocessing import StandardScaler

PERCENTIL_INF = 0.01
PERCENTIL_SUP = 0.99
VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")
CARPETA_DATOS = RAIZ / "01_datos"
CARPETA_MODELOS = RAIZ / "02_modelos"
CARPETA_MODELOS.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Variables de la Actividad 15
# ----------------------------------------------------------
df = pd.read_parquet(CARPETA_DATOS / "variables_muestra.parquet")
print("Filas:", len(df))
es_ajuste = (df["grupo"] == "entrenamiento") & (df["bloque"] == "entrenamiento")
print("Filas para ajustar (empresas y bloque de entrenamiento):", es_ajuste.sum())

# ----------------------------------------------------------
# 2. Límites de recorte: SOLO con el subconjunto de ajuste
# ----------------------------------------------------------
limites = df.loc[es_ajuste, VARIABLES].quantile([PERCENTIL_INF, PERCENTIL_SUP]).T
limites.columns = ["limite_inferior", "limite_superior"]
print("\n=== LÍMITES DE RECORTE (percentiles 1 y 99 del entrenamiento) ===")
print(limites)

# Cuánto se recorta en cada combinación de grupo y bloque (%)
fuera = (df[VARIABLES].lt(limites["limite_inferior"]) | df[VARIABLES].gt(limites["limite_superior"]))
print("\n=== % DE VALORES RECORTADOS POR GRUPO Y BLOQUE ===")
print((fuera.groupby([df["grupo"], df["bloque"]], observed=True).mean() * 100).round(2))

recortadas = df[VARIABLES].clip(lower=limites["limite_inferior"], upper=limites["limite_superior"], axis=1)

# ----------------------------------------------------------
# 3. Escalado: media 0 y desviación 1, ajustado SOLO con el subconjunto de ajuste
# ----------------------------------------------------------
escalador = StandardScaler()
escalador.fit(recortadas[es_ajuste])
print("\n=== PARÁMETROS DEL ESCALADOR (aprendidos en entrenamiento) ===")
print(pd.DataFrame({"media": escalador.mean_, "desviacion": escalador.scale_}, index=VARIABLES))

COLUMNAS_Z = ["z_" + v for v in VARIABLES]
escaladas = pd.DataFrame(escalador.transform(recortadas), columns=COLUMNAS_Z, index=df.index)

print("\n=== MEDIA DE LAS VARIABLES ESCALADAS POR GRUPO Y BLOQUE ===")
print(escaladas.groupby([df["grupo"], df["bloque"]], observed=True).mean())
print("\n=== DESVIACIÓN DE LAS VARIABLES ESCALADAS POR GRUPO Y BLOQUE ===")
print(escaladas.groupby([df["grupo"], df["bloque"]], observed=True).std())

# ----------------------------------------------------------
# 4. Guardar datos preparados, límites y escalador
# ----------------------------------------------------------
salida = pd.concat([df, escaladas], axis=1)      # conserva las variables originales para interpretar
salida.to_parquet(CARPETA_DATOS / "variables_preparadas.parquet", index=False)
limites.to_csv(CARPETA_MODELOS / "limites_recorte.csv", index_label="variable")
joblib.dump(escalador, CARPETA_MODELOS / "escalador.joblib")
print("\nGuardado: variables_preparadas.parquet,", len(salida), "filas;"
      " 02_modelos/limites_recorte.csv y 02_modelos/escalador.joblib")

Mounted at /content/drive
Filas: 2426275
Filas para ajustar (empresas y bloque de entrenamiento): 815829

=== LÍMITES DE RECORTE (percentiles 1 y 99 del entrenamiento) ===
                    limite_inferior  limite_superior
volatilidad_21                0.131            1.756
cambio_volatilidad           -0.793            0.500
rendimiento_21               -0.523            0.414
caida_maximo_63              -0.595            0.000
volumen_anormal              -0.638            1.318

=== % DE VALORES RECORTADOS POR GRUPO Y BLOQUE ===
                             volatilidad_21  cambio_volatilidad  rendimiento_21  caida_maximo_63  volumen_anormal
grupo         bloque                                                                                             
entrenamiento entrenamiento           2.000               2.000           2.000            1.000            2.000
              prueba                  1.370               1.700           1.040            0.440            1.610
  